In [1]:
# ============================================================
# 04 - RETRIEVAL
# Chunk 1: Imports and Project Paths
# ============================================================

import json
import pickle
from pathlib import Path

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully")


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()

# If Jupyter is running from the notebooks folder,
# move one level up to the project root.
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
RETRIEVAL_DIR = PROJECT_DIR / "data" / "retrieval"

RETRIEVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject:", PROJECT_DIR)
print("Vector Store:", VECTOR_DIR)
print("Retrieval Output:", RETRIEVAL_DIR)

c:\Users\Dell\anaconda3\envs\asl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully

Project: d:\Sem VII\AI\Project\Implementation\Hybrid-RAG-Project
Vector Store: d:\Sem VII\AI\Project\Implementation\Hybrid-RAG-Project\data\vector_store
Retrieval Output: d:\Sem VII\AI\Project\Implementation\Hybrid-RAG-Project\data\retrieval


In [2]:
# ============================================================
# Chunk 2: Verify Vector Store Files
# ============================================================

required_files = [
    "naive_rag.index",
    "naive_rag_metadata.pkl",
    "sentence_window.index",
    "sentence_window_metadata.pkl",
    "parent_child.index",
    "parent_child_metadata.pkl",
    "embedding_info.json"
]

print("Checking vector-store files...\n")

for filename in required_files:

    file_path = VECTOR_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required vector-store file not found: {file_path}"
        )

    print(f"✓ {filename}")

print("\nAll vector-store files are available.")

Checking vector-store files...

✓ naive_rag.index
✓ naive_rag_metadata.pkl
✓ sentence_window.index
✓ sentence_window_metadata.pkl
✓ parent_child.index
✓ parent_child_metadata.pkl
✓ embedding_info.json

All vector-store files are available.


In [3]:
# ============================================================
# Chunk 3: Load Embedding Configuration
# ============================================================

with open(
    VECTOR_DIR / "embedding_info.json",
    "r",
    encoding="utf-8"
) as f:

    embedding_info = json.load(f)


print("Embedding configuration:")
print(json.dumps(embedding_info, indent=4))

Embedding configuration:
{
    "model": "all-MiniLM-L6-v2",
    "dimension": 384,
    "device": "cpu",
    "similarity": "cosine_similarity",
    "faiss_index_type": "IndexFlatIP"
}


In [4]:
# ============================================================
# Chunk 4: Load Embedding Model
# ============================================================

MODEL_NAME = embedding_info["model"]

print("Loading embedding model...")
print("Model:", MODEL_NAME)

embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("\nEmbedding model loaded successfully!")
print("Device:", embedding_model.device)

Loading embedding model...
Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 764.25it/s]



Embedding model loaded successfully!
Device: cpu


In [5]:
# ============================================================
# Chunk 5: Load FAISS Indexes
# ============================================================

naive_index = faiss.read_index(
    str(VECTOR_DIR / "naive_rag.index")
)

sentence_window_index = faiss.read_index(
    str(VECTOR_DIR / "sentence_window.index")
)

parent_child_index = faiss.read_index(
    str(VECTOR_DIR / "parent_child.index")
)


print("FAISS indexes loaded successfully!\n")

print("Naive RAG vectors:",
      naive_index.ntotal)

print("Sentence-Window vectors:",
      sentence_window_index.ntotal)

print("Parent-Child vectors:",
      parent_child_index.ntotal)

FAISS indexes loaded successfully!

Naive RAG vectors: 5
Sentence-Window vectors: 105
Parent-Child vectors: 38


In [8]:
# ============================================================
# Chunk 6: Load Retrieval Metadata
# ============================================================

with open(
    VECTOR_DIR / "naive_rag_metadata.pkl",
    "rb"
) as f:

    naive_metadata = pickle.load(f)


with open(
    VECTOR_DIR / "sentence_window_metadata.pkl",
    "rb"
) as f:

    sentence_window_metadata = pickle.load(f)


with open(
    VECTOR_DIR / "parent_child_metadata.pkl",
    "rb"
) as f:

    parent_child_metadata = pickle.load(f)


print("Metadata loaded successfully!\n")

print("Naive metadata:",
      len(naive_metadata))

print("Sentence-Window metadata:",
      len(sentence_window_metadata))

print("Parent-Child metadata:",
      len(parent_child_metadata))

Metadata loaded successfully!

Naive metadata: 5
Sentence-Window metadata: 105
Parent-Child metadata: 38


In [9]:
# ============================================================
# Chunk 7: Query Embedding Function
# ============================================================

def create_query_embedding(query):
    """
    Convert a query into a normalized embedding.
    """

    embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    embedding = np.asarray(
        embedding,
        dtype="float32"
    )

    return embedding


print("Query embedding function created successfully!")

Query embedding function created successfully!


In [10]:
# ============================================================
# Chunk 8: Generic FAISS Retrieval Function
# ============================================================

def retrieve(
    query,
    index,
    metadata,
    k=5
):
    """
    Retrieve top-k results from a FAISS index.
    """

    query_embedding = create_query_embedding(
        query
    )

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx == -1:
            continue

        results.append({
            "rank": rank,
            "score": float(score),
            "index": int(idx),
            "metadata": metadata[idx]
        })

    return results


print("Generic retrieval function created successfully!")

Generic retrieval function created successfully!


In [11]:
# ============================================================
# Chunk 9: Retrieval Text Helper
# ============================================================

def get_result_text(
    result,
    retrieval_method
):
    """
    Extract the text from a retrieval result.
    """

    metadata = result["metadata"]

    if retrieval_method == "naive":

        return metadata["text"]

    elif retrieval_method == "sentence_window":

        return metadata["window_text"]

    elif retrieval_method == "parent_child":

        return metadata["child_text"]

    else:

        raise ValueError(
            f"Unknown retrieval method: {retrieval_method}"
        )


print("Result text helper created successfully!")

Result text helper created successfully!


In [12]:
# ============================================================
# Chunk 10: Naive RAG Retrieval
# ============================================================

query = "Who is eligible for PMAY-U?"

TOP_K = 5

naive_results = retrieve(
    query=query,
    index=naive_index,
    metadata=naive_metadata,
    k=TOP_K
)

print("=" * 70)
print("NAIVE RAG RETRIEVAL")
print("=" * 70)

print("Query:", query)

for result in naive_results:

    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.4f}"
    )

    print(
        "Section:",
        result["metadata"]["section"]
    )

    print(
        "Text:",
        result["metadata"]["text"][:500]
    )

NAIVE RAG RETRIEVAL
Query: Who is eligible for PMAY-U?

Rank 1 | Score: 0.6968
Section: application_process
Text: 1. Eligible beneficiaries can apply for PMAY-U through the applicable application process provided by the government.

2. Applications may be submitted through the online system or through the concerned Urban Local Body (ULB) or other designated agencies.

3. The applicant must provide the required personal, family, income and other details along with the necessary documents.

4. The application is verified by the concerned authorities before the beneficiary is considered for assistance under th

Rank 2 | Score: 0.5531
Section: details
Text: Pradhan Mantri Awas Yojana - Urban (PMAY-U) is a scheme of the Ministry of Housing & Urban Affairs (MoHUA), Government of India.

The scheme was launched to address the urban housing shortage among economically weaker sections (EWS), low income groups (LIG), middle income groups (MIG), including slum dwellers, by ensuring a pucca house 

In [14]:
# ============================================================
# Chunk 11: Sentence-Window Retrieval
# ============================================================

sentence_window_results = retrieve(
    query=query,
    index=sentence_window_index,
    metadata=sentence_window_metadata,
    k=TOP_K
)

print("=" * 70)
print("SENTENCE-WINDOW RETRIEVAL")
print("=" * 70)

print("Query:", query)

for result in sentence_window_results:

    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.4f}"
    )

    print(
        "Section:",
        result["metadata"]["section"]
    )

    print(
        "Text:",
        result["metadata"]["window_text"][:500]
    )

SENTENCE-WINDOW RETRIEVAL
Query: Who is eligible for PMAY-U?

Rank 1 | Score: 0.7486
Section: APPLICATION PROCESS
Text: 1. Eligible beneficiaries can apply for PMAY-U through the applicable application process provided by the government.

Rank 2 | Score: 0.7448
Section: APPLICATION PROCESS
Text: 1. Eligible beneficiaries can apply for PMAY-U through the applicable application process provided by the government. 2.

Rank 3 | Score: 0.7256
Section: APPLICATION PROCESS
Text: Eligible beneficiaries can apply for PMAY-U through the applicable application process provided by the government. 2. Applications may be submitted through the online system or through the concerned Urban Local Body (ULB) or other designated agencies.

Rank 4 | Score: 0.6519
Section: FREQUENTLY ASKED QUESTIONS
Text: Preference will be given to differently-abled persons, senior citizens, SCs, STs, OBCs, Minority, single women, transgender and other weaker & vulnerable sections of the society. 4. Will I still be eligibl

In [15]:
# ============================================================
# Chunk 12: Parent-Child Retrieval
# ============================================================

parent_child_results = retrieve(
    query=query,
    index=parent_child_index,
    metadata=parent_child_metadata,
    k=TOP_K
)

print("=" * 70)
print("PARENT-CHILD RETRIEVAL")
print("=" * 70)

print("Query:", query)

for result in parent_child_results:

    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']:.4f}"
    )

    print(
        "Section:",
        result["metadata"]["section"]
    )

    print(
        "Child:",
        result["metadata"]["child_text"][:500]
    )

PARENT-CHILD RETRIEVAL
Query: Who is eligible for PMAY-U?

Rank 1 | Score: 0.7448
Section: APPLICATION PROCESS
Child: 1. Eligible beneficiaries can apply for PMAY-U through the applicable application process provided by the government. 2.

Rank 2 | Score: 0.6232
Section: FREQUENTLY ASKED QUESTIONS
Child: 4. Will I still be eligible to apply for government’s ongoing or upcoming schemes if I apply for PMAY-U? The benefits of all existing and future rural schemes will not be denied to a beneficiary solely on the grounds that she/he has availed a house under PMAY(U).

Rank 3 | Score: 0.5988
Section: APPLICATION PROCESS
Child: 4. The application is verified by the concerned authorities before the beneficiary is considered for assistance under the applicable component of PMAY-U.

Rank 4 | Score: 0.5622
Section: DETAILS
Child: Pradhan Mantri Awas Yojana - Urban (PMAY-U) is a scheme of the Ministry of Housing & Urban Affairs (MoHUA), Government of India. The scheme was launched to address the 

In [16]:
# ============================================================
# Chunk 13: Compare Retrieval Methods
# ============================================================

print("=" * 80)
print("RETRIEVAL COMPARISON")
print("=" * 80)

print("\nQuery:", query)


# ------------------------------------------------------------
# Naive
# ------------------------------------------------------------

print("\n\n### NAIVE RAG ###")

for result in naive_results:

    section = result["metadata"]["section"]

    print(
        f"Rank {result['rank']} | "
        f"Score {result['score']:.4f} | "
        f"Section: {section}"
    )


# ------------------------------------------------------------
# Sentence Window
# ------------------------------------------------------------

print("\n\n### SENTENCE-WINDOW ###")

for result in sentence_window_results:

    section = result["metadata"]["section"]

    print(
        f"Rank {result['rank']} | "
        f"Score {result['score']:.4f} | "
        f"Section: {section}"
    )


# ------------------------------------------------------------
# Parent Child
# ------------------------------------------------------------

print("\n\n### PARENT-CHILD ###")

for result in parent_child_results:

    section = result["metadata"]["section"]

    print(
        f"Rank {result['rank']} | "
        f"Score {result['score']:.4f} | "
        f"Section: {section}"
    )

RETRIEVAL COMPARISON

Query: Who is eligible for PMAY-U?


### NAIVE RAG ###
Rank 1 | Score 0.6968 | Section: application_process
Rank 2 | Score 0.5531 | Section: details
Rank 3 | Score 0.3739 | Section: documents_required
Rank 4 | Score 0.3056 | Section: benefits
Rank 5 | Score 0.2786 | Section: eligibility


### SENTENCE-WINDOW ###
Rank 1 | Score 0.7486 | Section: APPLICATION PROCESS
Rank 2 | Score 0.7448 | Section: APPLICATION PROCESS
Rank 3 | Score 0.7256 | Section: APPLICATION PROCESS
Rank 4 | Score 0.6519 | Section: FREQUENTLY ASKED QUESTIONS
Rank 5 | Score 0.6232 | Section: FREQUENTLY ASKED QUESTIONS


### PARENT-CHILD ###
Rank 1 | Score 0.7448 | Section: APPLICATION PROCESS
Rank 2 | Score 0.6232 | Section: FREQUENTLY ASKED QUESTIONS
Rank 3 | Score 0.5988 | Section: APPLICATION PROCESS
Rank 4 | Score 0.5622 | Section: DETAILS
Rank 5 | Score 0.5506 | Section: FREQUENTLY ASKED QUESTIONS


In [17]:
# ============================================================
# Chunk 14: Test Query Set
# ============================================================

test_queries = [
    "Who is eligible for PMAY-U?",
    "What are the benefits of PMAY-U?",
    "What documents are required to apply for PMAY-U?",
    "How can I apply for PMAY-U?",
    "Can a family that already owns a pucca house apply for PMAY-U?"
]

print("Test queries:")
print()

for i, q in enumerate(test_queries, 1):

    print(f"{i}. {q}")

Test queries:

1. Who is eligible for PMAY-U?
2. What are the benefits of PMAY-U?
3. What documents are required to apply for PMAY-U?
4. How can I apply for PMAY-U?
5. Can a family that already owns a pucca house apply for PMAY-U?


In [19]:
# ============================================================
# Chunk 15: Run Query Evaluation Set
# ============================================================

all_retrieval_results = []

for query_id, query_text in enumerate(
    test_queries,
    start=1
):

    print(
        f"\nProcessing Query {query_id}: "
        f"{query_text}"
    )

    naive = retrieve(
        query_text,
        naive_index,
        naive_metadata,
        k=TOP_K
    )

    sentence_window = retrieve(
        query_text,
        sentence_window_index,
        sentence_window_metadata,
        k=TOP_K
    )

    parent_child = retrieve(
        query_text,
        parent_child_index,
        parent_child_metadata,
        k=TOP_K
    )

    all_retrieval_results.append({

        "query_id": query_id,

        "query": query_text,

        "naive": naive,

        "sentence_window": sentence_window,

        "parent_child": parent_child
    })


print("\nAll queries processed successfully!")
print(
    "Total queries:",
    len(all_retrieval_results)
)


Processing Query 1: Who is eligible for PMAY-U?

Processing Query 2: What are the benefits of PMAY-U?

Processing Query 3: What documents are required to apply for PMAY-U?

Processing Query 4: How can I apply for PMAY-U?

Processing Query 5: Can a family that already owns a pucca house apply for PMAY-U?

All queries processed successfully!
Total queries: 5


In [20]:
# ============================================================
# Chunk 16: Retrieval Summary
# ============================================================

for item in all_retrieval_results:

    print("\n" + "=" * 80)

    print(
        f"Query {item['query_id']}: "
        f"{item['query']}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Naive
    # --------------------------------------------------------

    print("\nNaive RAG:")

    for result in item["naive"]:

        print(
            f"  {result['rank']}. "
            f"{result['metadata']['section']} "
            f"({result['score']:.4f})"
        )


    # --------------------------------------------------------
    # Sentence Window
    # --------------------------------------------------------

    print("\nSentence-Window:")

    for result in item["sentence_window"]:

        print(
            f"  {result['rank']}. "
            f"{result['metadata']['section']} "
            f"({result['score']:.4f})"
        )


    # --------------------------------------------------------
    # Parent Child
    # --------------------------------------------------------

    print("\nParent-Child:")

    for result in item["parent_child"]:

        print(
            f"  {result['rank']}. "
            f"{result['metadata']['section']} "
            f"({result['score']:.4f})"
        )


Query 1: Who is eligible for PMAY-U?

Naive RAG:
  1. application_process (0.6968)
  2. details (0.5531)
  3. documents_required (0.3739)
  4. benefits (0.3056)
  5. eligibility (0.2786)

Sentence-Window:
  1. APPLICATION PROCESS (0.7486)
  2. APPLICATION PROCESS (0.7448)
  3. APPLICATION PROCESS (0.7256)
  4. FREQUENTLY ASKED QUESTIONS (0.6519)
  5. FREQUENTLY ASKED QUESTIONS (0.6232)

Parent-Child:
  1. APPLICATION PROCESS (0.7448)
  2. FREQUENTLY ASKED QUESTIONS (0.6232)
  3. APPLICATION PROCESS (0.5988)
  4. DETAILS (0.5622)
  5. FREQUENTLY ASKED QUESTIONS (0.5506)

Query 2: What are the benefits of PMAY-U?

Naive RAG:
  1. details (0.4954)
  2. application_process (0.4415)
  3. documents_required (0.3409)
  4. benefits (0.2002)
  5. eligibility (0.1555)

Sentence-Window:
  1. FREQUENTLY ASKED QUESTIONS (0.6650)
  2. FREQUENTLY ASKED QUESTIONS (0.5695)
  3. FREQUENTLY ASKED QUESTIONS (0.5665)
  4. DETAILS (0.5528)
  5. DETAILS (0.5279)

Parent-Child:
  1. FREQUENTLY ASKED QUESTION

In [21]:
# ============================================================
# Chunk 17: Prepare JSON-Safe Retrieval Results
# ============================================================

def simplify_results(
    results,
    retrieval_method
):

    simplified = []

    for result in results:

        metadata = result["metadata"]

        simplified.append({

            "rank": result["rank"],

            "score": result["score"],

            "index": result["index"],

            "method": retrieval_method,

            "section": metadata.get(
                "section",
                ""
            ),

            "text": get_result_text(
                result,
                retrieval_method
            )
        })

    return simplified


json_results = []

for item in all_retrieval_results:

    json_results.append({

        "query_id": item["query_id"],

        "query": item["query"],

        "naive": simplify_results(
            item["naive"],
            "naive"
        ),

        "sentence_window": simplify_results(
            item["sentence_window"],
            "sentence_window"
        ),

        "parent_child": simplify_results(
            item["parent_child"],
            "parent_child"
        )
    })


print("JSON-safe retrieval results prepared!")

JSON-safe retrieval results prepared!


In [22]:
# ============================================================
# Chunk 18: Save Retrieval Results
# ============================================================

RETRIEVAL_RESULTS_PATH = (
    RETRIEVAL_DIR /
    "retrieval_results.json"
)

with open(
    RETRIEVAL_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_results,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Retrieval results saved successfully!")
print(
    "File:",
    RETRIEVAL_RESULTS_PATH
)

Retrieval results saved successfully!
File: d:\Sem VII\AI\Project\Implementation\Hybrid-RAG-Project\data\retrieval\retrieval_results.json


In [23]:
# ============================================================
# Chunk 19: Final Verification
# ============================================================

print("=" * 70)
print("04 - RETRIEVAL COMPLETE")
print("=" * 70)

print("\nEmbedding model:")
print(" ", MODEL_NAME)

print("\nVector stores:")
print(
    " Naive RAG:",
    naive_index.ntotal,
    "vectors"
)

print(
    " Sentence-Window:",
    sentence_window_index.ntotal,
    "vectors"
)

print(
    " Parent-Child:",
    parent_child_index.ntotal,
    "vectors"
)

print("\nTest queries:")
print(" ", len(test_queries))

print("\nRetrieval results:")
print(
    " ",
    RETRIEVAL_RESULTS_PATH
)

print("\nStatus: SUCCESS")

04 - RETRIEVAL COMPLETE

Embedding model:
  all-MiniLM-L6-v2

Vector stores:
 Naive RAG: 5 vectors
 Sentence-Window: 105 vectors
 Parent-Child: 38 vectors

Test queries:
  5

Retrieval results:
  d:\Sem VII\AI\Project\Implementation\Hybrid-RAG-Project\data\retrieval\retrieval_results.json

Status: SUCCESS
